# 🤖 Mountain Risk Assessment Agent
Agente conversacional desarrollado con LangChain y Llama 3.3 70B (GroqCloud) que evalúa
riesgo en excursiones de montaña mediante lenguaje natural y genera recomendaciones personalizadas como Guía de Montaña

In [1]:
# limpiar todo
!pip uninstall -y langchain langchain-core langchain-community langchain-groq langgraph

# instalar stack compatible moderno
!pip install -q \
langchain==0.2.5 \
langchain-core==0.2.9 \
langchain-community==0.2.4 \
langchain-groq==0.1.4 \
langgraph==0.0.55 \
astral \
requests

Found existing installation: langchain 0.2.5
Uninstalling langchain-0.2.5:
  Successfully uninstalled langchain-0.2.5
Found existing installation: langchain-core 0.2.9
Uninstalling langchain-core-0.2.9:
  Successfully uninstalled langchain-core-0.2.9
Found existing installation: langchain-community 0.2.4
Uninstalling langchain-community-0.2.4:
  Successfully uninstalled langchain-community-0.2.4
Found existing installation: langchain-groq 0.1.4
Uninstalling langchain-groq-0.1.4:
  Successfully uninstalled langchain-groq-0.1.4
Found existing installation: langgraph 0.0.55
Uninstalling langgraph-0.0.55:
  Successfully uninstalled langgraph-0.0.55
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph-sdk 0.4.2 requires langchain-core<2,>=1.4.0, but you have langchain-core 0.2.9 which is incompatible.
langgraph-checkpoint 4.1.1 requires langchain-core>=0.2.38, 

In [2]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/hiking_agent_v3

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/hiking_agent_v3


In [3]:
# Importación de tools y risk engine
from tools.trails_tool import get_trail_by_id, get_all_trails
from tools.weather_tool import get_weather
from tools.daylight_tool import get_daylight_hours
from tools.tourism_tool import get_tourism_description

from core.risk_engine import classify_experience, compute_weighted_risk, risk_category

# Construir contexto determinístico
def build_risk_context(trail_id: int, date: str):
    trail = get_trail_by_id(trail_id)
    if not trail:
        raise ValueError(f"No se encontró sendero con id {trail_id}")

    weather = get_weather(date)
    if weather is None or "temp_c" not in weather:
        raise ValueError(f"No se pudo obtener clima para la fecha {date}")

    light = get_daylight_hours(date)
    if light is None or "daylight_hours" not in light:
        raise ValueError(f"No se pudieron obtener horas de luz para la fecha {date}")

    return {"trail": trail, "weather": weather, "light": light}

Agente 1 - Trip Planner

In [4]:
# ----------------------------------------------
# Agente 1
# Texto libre → datos estructurados
# Modelo: Llama 3.1 (Groq)
# ----------------------------------------------

import os
import json
from google.colab import userdata
from langchain_groq import ChatGroq

from tools.trails_tool import get_all_trails, get_trail_by_id

# API key
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

# LLM parser
llm_parser = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)


def get_user_inputs_from_text(user_input: str):

    # 1️⃣ Obtener senderos desde la tool
    trails = get_all_trails()

    # 2️⃣ Crear lista dinámica para el prompt
    trails_text = "\n".join(
        [f"{t['id']} - {t['name']}" for t in trails]
    )

    # 3️⃣ Prompt al LLM
    prompt = f"""
Convertí este mensaje de trekking en JSON válido.

Senderos disponibles:
{trails_text}

Formato de salida:

{{
 "trail_id": number,
 "fecha_salida": "YYYY-MM-DD",
 "hora_salida": "HH:MM",
 "q1_12km": true/false,
 "q2_20km": true/false,
 "q3_fisica_regular": true/false
}}

Reglas:
- Si el usuario dice "nunca hice", "no hice" o "jamás hice", el valor debe ser false.
- Elegí el trail_id correcto según el nombre del sendero.
- Respondé SOLO con JSON válido.
- No agregues texto adicional.

Mensaje:
{user_input}
"""

    # 4️⃣ Llamada al LLM
    response = llm_parser.invoke(prompt)

    text_output = response.content.strip()

    # limpiar markdown
    text_output = text_output.replace("```json", "").replace("```", "").strip()

    # 5️⃣ Parsear JSON
    try:
        parsed = json.loads(text_output)
    except:
        raise ValueError(f"Respuesta inválida:\n{text_output}")

    # 6️⃣ Validación real del sendero usando tool
    trail = get_trail_by_id(parsed["trail_id"])

    if trail is None:
        raise ValueError("Sendero no válido")

    return parsed

In [5]:
# ----------------------------------------------
# Construcción de contexto de riesgo
# ----------------------------------------------

def build_risk_context(trail_id: int, date: str):

    trail = get_trail_by_id(trail_id)

    if not trail:
        raise ValueError(f"No se encontró sendero con id {trail_id}")

    weather = get_weather(date)

    if weather is None or "temp_c" not in weather:
        raise ValueError(f"No se pudo obtener clima para la fecha {date}")

    light = get_daylight_hours(date)

    if light is None or "daylight_hours" not in light:
        raise ValueError(f"No se pudieron obtener horas de luz para la fecha {date}")

    return {
        "trail": trail,
        "weather": weather,
        "light": light
    }


# ----------------------------------------------
# Motor de evaluación de riesgo
# ----------------------------------------------

def run_assessment(user_data: dict):

    # 1️⃣ Nivel de experiencia del usuario
    experience_level = classify_experience(
        user_data.get("q1_12km", False),
        user_data.get("q2_20km", False),
        user_data.get("q3_fisica_regular", False)
    )

    # 2️⃣ Contexto determinístico
    risk_context = build_risk_context(
        trail_id=user_data["trail_id"],
        date=user_data["fecha_salida"]
    )

    # 3️⃣ Score de riesgo
    score = compute_weighted_risk(
        trail=risk_context["trail"],
        level=experience_level,
        weather_data=risk_context["weather"],
        light_data=risk_context["light"],
        start_time=user_data["hora_salida"]
    )

    # 4️⃣ Categoría
    category = risk_category(score)

    return {
        "experience_level": experience_level,
        "score": score,
        "category": category,
        "trail_id": user_data["trail_id"],
        "fecha_salida": user_data["fecha_salida"],
        "hora_salida": user_data["hora_salida"]
    }

In [6]:
#---------------------------
# Definir tools
#---------------------------

from langchain_core.tools import tool

@tool
def trail_tool(trail_id: int) -> dict:
    """
    Return trail information for a given trail_id.
    """

    trail = get_trail_by_id(trail_id)

    if trail is None:
        return {"error": f"Trail {trail_id} not found"}

    return trail


@tool
def weather_tool(date: str) -> dict:
    """
    Return weather forecast for a given date.
    """

    weather = get_weather(date)

    if weather is None:
        return {"error": f"No weather data for {date}"}

    return weather


@tool
def daylight_tool(date: str) -> dict:
    """
    Return daylight hours for a given date.
    """

    daylight = get_daylight_hours(date)

    if daylight is None:
        return {"error": f"No daylight data for {date}"}

    return daylight


@tool
def tourism_tool(trail_name: str) -> str:
    """
    Return tourism description for a trail.
    """

    description = get_tourism_description(trail_name)

    if description is None:
        return "No tourism information available."

    return description


tools = [
    trail_tool,
    weather_tool,
    daylight_tool,
    tourism_tool
]

In [7]:
# LLM Groq para LangChain

import os
from google.colab import userdata
from langchain_groq import ChatGroq

# cargar API key
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

# crear modelo
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.3
)

In [8]:
# Nodo del agente ReAct con tools

from langchain.agents import initialize_agent, AgentType

agent_executor = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=False,
    max_iterations=10,
    handle_parsing_errors=True
)

def agent_node(state: dict):

    trail = get_trail_by_id(state["trail_id"])

    prompt = f"""
Sos un asistente especializado en seguridad para senderismo
en el Parque Nacional Nahuel Huapi.

Datos de la salida:

Sendero: {trail["name"]}
Distancia: {trail["distance_km"]} km
Dificultad: {trail["difficulty"]}

Fecha: {state["fecha_salida"]}
Hora salida: {state["hora_salida"]}

Nivel de experiencia del visitante: {state["experience_level"]}
Score de riesgo calculado: {state["score"]}
Categoría de riesgo: {state["category"]}

Herramientas disponibles:
- weather_tool → clima para una fecha
- daylight_tool → horas de luz
- tourism_tool → descripción turística

Podés usar herramientas si necesitás información adicional.

Generá la respuesta final con esta estructura:

Contexto turístico:
(explicación breve del sendero)

Evaluación de seguridad:
(análisis considerando experiencia y score de riesgo)

Recomendación final:
(consejos concretos para la salida)
"""

    result = agent_executor.invoke({"input": prompt})

    return {
        **state,
        "final_response": result["output"]
    }

/usr/local/lib/python3.12/dist-packages/langchain_core/_api/deprecation.py:139: LangChainDeprecationWarning: The function `initialize_agent` was deprecated in LangChain 0.1.0 and will be removed in 0.3.0. Use Use new agent constructor methods like create_react_agent, create_json_agent, create_structured_chat_agent, etc. instead.
  warn_deprecated(


In [14]:
from IPython.display import HTML, display
from datetime import datetime
import json, re

def normalize_date(user_data, user_message):
    fecha = user_data.get("fecha_salida")
    if not fecha:
        return user_data
    try:
        parsed_date = datetime.strptime(fecha, "%Y-%m-%d")
        today = datetime.now()
        meses = ["enero","febrero","marzo","abril","mayo","junio",
                 "julio","agosto","septiembre","octubre","noviembre","diciembre"]
        if parsed_date.month != today.month:
            if not any(m in user_message.lower() for m in meses):
                parsed_date = parsed_date.replace(year=today.year, month=today.month)
        if parsed_date.year < today.year:
            parsed_date = parsed_date.replace(year=today.year)
        user_data["fecha_salida"] = parsed_date.strftime("%Y-%m-%d")
    except:
        pass
    return user_data

def validate_user_data(user_data):
    missing_fields = []
    if not user_data.get("fecha_salida"):
        missing_fields.append("fecha")
    if not user_data.get("hora_salida") or user_data["hora_salida"] == "00:00":
        missing_fields.append("hora")
    if not any([user_data.get("q1_12km"), user_data.get("q2_20km"), user_data.get("q3_fisica_regular")]):
        missing_fields.append("experiencia")
    return missing_fields

def parse_experience(text: str):
    prompt = f"""
Convertí esta descripción de experiencia en JSON.
Texto: {text}
Formato: {{"q1_12km": true/false, "q2_20km": true/false, "q3_fisica_regular": true/false}}
Reglas:
- "nunca", "no" → false
- "algo", "un poco" → true en 12km
- "gym", "entreno", "activo" → física regular = true
- Respondé SOLO JSON válido
"""
    response = llm_parser.invoke(prompt)
    output = response.content.strip().replace("```json", "").replace("```", "").strip()
    try:
        return json.loads(output)
    except:
        pass
    try:
        json_match = re.search(r"\{.*\}", output, re.DOTALL)
        if json_match:
            return json.loads(json_match.group())
    except:
        pass
    return {"q1_12km": False, "q2_20km": False, "q3_fisica_regular": False}

def run_agent_display(user_message):
    try:
        trails = get_all_trails()
        user_data = get_user_inputs_from_text(user_message)
        user_data = normalize_date(user_data, user_message)
        missing = validate_user_data(user_data)

        if missing:
            if "fecha" in missing:
                user_data["fecha_salida"] = input("¿Qué día querés salir? (YYYY-MM-DD): ").strip()
            if "hora" in missing:
                user_data["hora_salida"] = input("¿A qué hora pensás arrancar? (HH:MM): ").strip()
            if "experiencia" in missing:
                exp_text = input("Contame tu experiencia en trekking y estado físico: ").strip()
                user_data.update(parse_experience(exp_text))

        risk_result = run_assessment(user_data)
        state = {**user_data, **risk_result}
        result = agent_node(state)
        recomendacion_raw = result.get("final_response", "Sin recomendación disponible.")
        emoji = {"BAJO": "🟢", "MEDIO": "🟡", "ALTO": "🔴"}.get(state.get("category", ""), "⚪")

        # Separar secciones por palabras clave
        iconos = {
            "Contexto turístico:": "🗺️",
            "Evaluación de seguridad:": "📊",
            "Recomendación final:": "✅"
        }
        for seccion in iconos:
            if seccion in recomendacion_raw:
                recomendacion_raw = recomendacion_raw.replace(seccion, f"|||{seccion}")

        partes = [p.strip() for p in recomendacion_raw.split("|||") if p.strip()]

        bloques_html = ""
        for parte in partes:
            titulo = next((k for k in iconos if parte.startswith(k)), None)
            if titulo:
                icono = iconos[titulo]
                cuerpo = parte[len(titulo):].strip()
                bloques_html += f"""
<div style="margin-bottom:20px">
  <p style="color:#89b4fa;font-weight:bold;margin:0 0 6px 0">{icono} {titulo}</p>
  <p style="color:#cdd6f4;line-height:1.8;margin:0">{cuerpo}</p>
</div>"""
            else:
                bloques_html += f"<p style='color:#cdd6f4;line-height:1.8'>{parte}</p>"

        html = f"""
<div style="font-family:sans-serif;max-width:700px;margin:auto;background:#1e1e2e;border-radius:12px;padding:24px">
  <h2 style="color:#cdd6f4;margin-top:0">🏔️ Mountain Risk Assessment</h2>
  <div style="background:#313244;border-radius:8px;padding:16px;margin-bottom:16px">
    <h3 style="color:#cdd6f4;margin-top:0">{emoji} Riesgo: {state.get('category','-')}</h3>
    <table style="width:100%;border-collapse:collapse;color:#cdd6f4">
      <tr style="border-bottom:1px solid #45475a">
        <td style="padding:8px 0"><b>Sendero</b></td>
        <td style="padding:8px 0">{state.get('trail_id','-')}</td>
      </tr>
      <tr style="border-bottom:1px solid #45475a">
        <td style="padding:8px 0"><b>Fecha</b></td>
        <td style="padding:8px 0">{state.get('fecha_salida','-')}</td>
      </tr>
      <tr style="border-bottom:1px solid #45475a">
        <td style="padding:8px 0"><b>Hora</b></td>
        <td style="padding:8px 0">{state.get('hora_salida','-')}</td>
      </tr>
      <tr style="border-bottom:1px solid #45475a">
        <td style="padding:8px 0"><b>Experiencia</b></td>
        <td style="padding:8px 0">{state.get('experience_level','-')}</td>
      </tr>
      <tr>
        <td style="padding:8px 0"><b>Score</b></td>
        <td style="padding:8px 0">{round(state.get('score',0),2)}</td>
      </tr>
    </table>
  </div>
  <div style="background:#313244;border-left:4px solid #89b4fa;padding:16px;border-radius:4px">
    <h3 style="color:#cdd6f4;margin-top:0">🧭 Recomendación</h3>
    {bloques_html}
  </div>
</div>
"""
        display(HTML(html))

    except Exception as e:
        display(HTML(f"<div style='color:red;font-family:sans-serif'>❌ Error: {e}</div>"))

# ----------------------------------------------
# Ejecutar
# ----------------------------------------------
user_message = input("Describí tu plan (ej: 'Frey mañana a las 10'): ")
run_agent_display(user_message)

Describí tu plan (ej: 'Frey mañana a las 10'): Frey mañana 8 am
Contame tu experiencia en trekking y estado físico: poca experiencia, hago ejercicio físico regularmente


Sendero,2
Fecha,2026-06-30
Hora,08:00
Experiencia,intermedio
Score,35.5
